# Spam Detector — Exploration Notebook

> **Note:** This notebook is kept as a record of the original exploration.
> The project has since been restructured into reusable scripts and a
> Streamlit app. For the current workflow use:
>
> - `python train.py` — train, evaluate, tune, and save the model
> - `python predict.py "message"` — classify a message
> - `streamlit run app.py` — interactive browser app
>
> The cell below mirrors that pipeline in notebook form.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [ ]:
# -----------------------------------
# 1. Load the dataset
# -----------------------------------

data = pd.read_csv("spam or ham dataset.csv", encoding='latin-1')
data = data[["v1", "v2"]]
data.columns = ["label", "message"]

print(data.head())
print(data.columns)


# -----------------------------------
# 2. Select columns
# -----------------------------------

X = data["message"]
y = data["label"]


# -----------------------------------
# 3. Convert labels into numbers
# ham  = 0
# spam = 1
# -----------------------------------

y = y.map({
    "ham": 0,
    "spam": 1
})


# -----------------------------------
# 4. Split dataset
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# -----------------------------------
# 5. Convert text into numerical data
# -----------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df = 2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


# -----------------------------------
# 6. Create and train the model
# -----------------------------------

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)


# -----------------------------------
# 7. Evaluate the model
# -----------------------------------

y_pred = model.predict(X_test_tfidf)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# -----------------------------------
# 8. Test a new message
# -----------------------------------

message = ["Congratulations! You have won a free prize. Click now!"]

message_tfidf = vectorizer.transform(message)
prediction = model.predict(message_tfidf)

if prediction[0] == 1:
    print("\nPrediction: SPAM")
else:
    print("\nPrediction: NOT SPAM")